### LoRA Fine-Tuning Demo

In this section, we demonstrate a very small example of parameter-efficient fine-tuning using **LoRA**.

Instead of updating all model parameters, LoRA adds a small number of trainable parameters on top of the original model.

This makes fine-tuning much lighter and more practical for workshop use.

In this demo, we will:

- use the same Qwen model from earlier workshops
- create a very small health-related instruction dataset
- run a short LoRA fine-tuning process
- compare behaviour before and after training

## What is LoRA?

LoRA (**Low-Rank Adaptation**) is a lightweight fine-tuning method for large language models.

Instead of updating all model parameters, LoRA:
- keeps the original model fixed
- adds a small number of trainable adapter weights
- fine-tunes only these smaller components

This makes training:
- faster
- cheaper
- more memory-efficient

In this workshop, we use LoRA as a simple way to adapt Qwen without full fine-tuning.

**Reference:** Hu et al. (2021), *LoRA: Low-Rank Adaptation of Large Language Models*.

In [1]:
import!pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 20.4 MB/s eta 0:00:00


In [2]:
#import libraries
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

## Step 2: Load the Base Model

We use the same Qwen model as before, but this time we load it in a form suitable for fine-tuning.

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("Base model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded.


## Step 3: Create a Tiny Training Dataset

For this demo, we use a very small set of health-related instruction–response examples.

The aim is not to fully train the model, but to show how LoRA can slightly adapt its behaviour.

In [4]:
examples = [
    {
        "text": """### Instruction:
What are common symptoms of flu?

### Response:
Common symptoms of flu include fever, cough, sore throat, body aches, fatigue, and chills."""
    },
    {
        "text": """### Instruction:
Can this assistant diagnose a disease?

### Response:
No. This assistant can provide general health information, but it does not diagnose diseases or replace professional medical advice."""
    },
    {
        "text": """### Instruction:
What should the assistant say if the answer is uncertain?

### Response:
If the answer is uncertain or not supported by the available context, the assistant should say: I don't know."""
    },
    {
        "text": """### Instruction:
Should this assistant provide treatment advice?

### Response:
No. This assistant should provide general health information only and should not give treatment advice."""
    }
]

dataset = Dataset.from_list(examples)
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

In [7]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
# `r`: size of the low-rank adapters  / Low-rank adapters are a compact way to update a model using fewer parameters
# `lora_alpha`: scaling factor for adapter updates
# `lora_dropout`: dropout applied during training
# `bias`: whether bias terms are trained
# `task_type`: the type of model task, here causal language modelling

In [6]:
#Training params
training_args = TrainingArguments(
    output_dir="./qwen_lora_demo",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

## Step 4: Train the Model with LoRA

We now use the `SFTTrainer` (Supervised Fine-Tuning Trainer) to apply LoRA on our small dataset.

This will:
- attach LoRA adapters to the model
- train only the adapter parameters
- leave the original model unchanged

In [8]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,3.061247
2,3.901789
3,2.381384
4,3.362559
5,2.344771
6,3.356326
7,3.817528
8,2.956268
9,2.312924
10,3.782525


TrainOutput(global_step=12, training_loss=3.1253665884335837, metrics={'train_runtime': 3.3926, 'train_samples_per_second': 3.537, 'train_steps_per_second': 3.537, 'total_flos': 929073328128.0, 'train_loss': 3.1253665884335837})

## Step 5: Test the Model After Fine-Tuning

We now test the model to see if its responses reflect the behaviour we introduced during training.

In [9]:
from transformers import pipeline

pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)

prompt = """### Instruction:
Can you diagnose diabetes?

### Response:
"""

output = pipe(prompt, max_new_tokens=100, do_sample=True)
print(output[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
Can you diagnose diabetes?

### Response:
Sure, I can help with that. To diagnose diabetes, we need to first determine if the person has any of the following symptoms:

1. Frequent urination
2. Increased thirst
3. Fatigue
4. Nausea or vomiting 
5. Unusual weight loss (more than what someone is used to)
6. Dribbling when drinking water
7. Skin that looks pale, dry, and thin
8. Slow-healing sores

If these symptoms are


This means the tiny LoRA demo did not fine-tune enough to change behaviour clearly.

Reasons:
dataset is too small

training is too light

Qwen’s original behaviour is still dominating